# GPU Solution

**Runtime:** Google Colab, T4 GPU runtime (CUDA 12.x)

This notebook produces the GPU reference timings for our benchmark. It is one of three
notebooks:

1. **`CPU_Baseline.ipynb`**: runs the pipeline on CPU and writes
   `results/timings_cpu.csv` plus `results/best_params.json`.
2. **`GPU_Solution.ipynb`** (this one): runs the same pipeline on GPU using
   cuDF / cuML / XGBoost-GPU, reuses the hyperparameters from the CPU notebook, and writes
   `results/timings_gpu.csv`.
3. **`Analysis.ipynb`**: joins the two CSVs and produces the speedup tables and figures
   used in the report.

Both timing notebooks are deliberately structured as mirrors of each other. Only the
execution engine changes, so any timing difference is attributable to hardware, not to
modelling choices.

## The underlying ML task

The dataset is the CFPB Consumer Complaints corpus, reused from a previous ML assignment
(`72782_Complaints_Notebook.ipynb`). The target is binary: did the consumer dispute the
company's response (`Consumer disputed?`, Yes/No)? We keep the original feature engineering
(target-encoded categoricals, narrative-derived text features, temporal flags,
response-quality flags) and the original two-model setup:

1. **Logistic Regression** (L2-regularised, class-balanced): the linear baseline.
2. **XGBoost** (histogram tree booster): the non-linear competitor.

What is new here is that we re-instrument the pipeline to measure wall-clock time at each
stage on GPU.

## How the notebook is organised

The notebook runs in two functionally distinct phases:

**Phase A, Hyperparameter search (Section 4).** A single, full-data CV search per model
on GPU. Best parameters from the CPU notebook are loaded from `results/best_params.json`
so both engines fit the same model; the GPU-side searches are timed for the context
baseline and as a correctness check (test-set AUC must be within 0.005 of the CPU result).
Section 4.2 additionally runs the XGBoost search twice, once with a pandas input and once
with a pre-loaded cuDF input, to isolate the cost of host-to-device (H2D) transfers.

**Phase B, Dataset-size sweep (Section 5).** The actual benchmark. For each of 5 fractions
(10%, 25%, 50%, 75%, 100% of the training set), we re-run the full pipeline on GPU and
time each phase separately. Each (fraction, phase) combination is repeated 3 times so the
analysis notebook can take the median and shake off scheduler noise. That gives
**3 runs × 5 fractions × 6 phases = 90 timing rows** in `results/timings_gpu.csv`.
A separate HPO sub-sweep (5.3) times `lr_cv` and `xgb_cv` and appends to the same CSV.

### What each phase measures

| Phase         | Stage         | What runs in this timed block                                                            |
|---------------|---------------|------------------------------------------------------------------------------------------|
| `fe_train`    | Feature eng.  | Fit + transform feature engineering on the training fold (target encoders learn from y) |
| `fe_test`     | Feature eng.  | Apply the fitted encoders to the test set (transform-only, no learning)                  |
| `lr_fit`      | Training      | Fit cuML Logistic Regression on the engineered training matrix                           |
| `xgb_fit`     | Training      | Fit XGBoost (`device='cuda'`) on the same engineered training matrix                     |
| `lr_predict`  | Inference     | `predict_proba` of cuML LR on the engineered test matrix                                 |
| `xgb_predict` | Inference     | `predict_proba` of XGBoost on the engineered test matrix                                 |
| `lr_cv`       | HPO (5.3)     | 5-fold CV over the LR grid (hand-rolled cuML LR), one run per fraction                   |
| `xgb_cv`      | HPO (5.3)     | 5-fold `RandomizedSearchCV` for XGBoost on GPU, one run per fraction                     |

## Notes on reproducibility

This notebook requires a **Colab T4 GPU runtime** (Runtime → Change runtime type → T4 GPU). RAPIDS (cuDF, cuML) is pre-installed on Colab GPU runtimes from 2025 onwards; §1.1 detects this and skips the pip install rather than risk a native library mismatch.

**Required inputs (must exist on Drive before running):**

- `/content/drive/MyDrive/complaints_training.csv` — same dataset path as `CPU_Baseline.ipynb`.
- `/content/drive/MyDrive/results/best_params.json` — produced by `CPU_Baseline.ipynb`. Hyperparameters are reused unchanged on GPU so the comparison measures the execution engine, not different models. **Run `CPU_Baseline.ipynb` first if this file is missing.**

**Outputs written by this notebook** (to `/content/drive/MyDrive/results/`):

- `timings_gpu.csv` — 95 rows, same schema as `timings_cpu.csv`. 90 rows from the fit/predict sweep + 5 HPO rows (3 `lr_cv` at 25/50/100% + 2 `xgb_cv` at 50/100%; 25% `xgb_cv` is omitted to limit total runtime).
- `best_params.json` — updated in-place with two new keys: `cv_context_seconds.xgb_randomizedsearch_gpu` (GPU CV time, pandas input) and `h2d_amortisation` (both CV times for the host-to-device amortisation experiment in §4.2).

§1 below mounts Drive and symlinks `results/` to the same Drive folder the CPU notebook wrote to. Total wall-clock is ~30–40 minutes on a T4; §4.2 (XGB RandomizedCV + H2D amortisation experiment) is the longest phase at ~24 minutes.

## Table of Contents

1. [Setup & Imports](#1-setup--imports)
   - 1.1 [RAPIDS Installation](#11-rapids-installation)
   - 1.2 [Imports](#12-imports)
2. [Data Loading & Train/Test Split](#2-data-loading--traintest-split)
3. [Feature Engineering](#3-feature-engineering)
   - 3.1 [CPU Reference Function (not benchmarked)](#31-cpu-reference-function-not-benchmarked)
   - 3.2 [GPU Feature Engineering](#32-gpu-feature-engineering-feature_engineering_gpu)
   - 3.3 [Correctness Verification: Feature Engineering](#33-correctness-verification-feature-engineering)
4. [Hyperparameter Search (Context Timing)](#4-hyperparameter-search-context-timing)
   - 4.1 [Logistic Regression: AUC Equivalence Check](#41-logistic-regression-auc-equivalence-check)
   - 4.2 [XGBoost: RandomizedSearchCV + H2D Amortisation](#42-xgboost-randomizedsearchcv--h2d-amortisation)
   - 4.3 [Persist GPU Context Timings](#43-persist-gpu-context-timings)
5. [Dataset-Size Sweep](#5-dataset-size-sweep)
   - 5.1 [Timing Utility](#51-timing-utility)
   - 5.2 [Fit/Predict Sweep](#52-fitpredict-sweep)
   - 5.3 [HPO Sweep](#53-hpo-sweep)
   - 5.4 [Sanity Check](#54-sanity-check)

## 1. Setup & Imports

### 1.1 RAPIDS Check / Installation

Colab GPU runtimes (2025+) pre-install RAPIDS. The cell below detects whether cuDF/cuML
are already present and **skips pip install** if so. Installing a different version on top
of the pre-installed one creates a native library mismatch (rmm, libraft) that silently
crashes the kernel.

In [ ]:
import subprocess, sys

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if 'NVIDIA' not in gpu_info.stdout:
    raise RuntimeError('No GPU detected — go to Runtime > Change runtime type > T4 GPU')
print('GPU OK:', [l for l in gpu_info.stdout.split('\n') if 'MiB' in l or 'T4' in l][:2])

# Colab (2025+) pre-installs RAPIDS on GPU runtimes.
# Attempting to pip-install a different version downgrades core libs (rmm, libraft)
# while leaving others (libcugraph, cudf-polars) at the newer version, causing a
# native library mismatch that silently crashes the kernel on first cuDF use.
# → only install if RAPIDS is genuinely absent.
rapids_check = subprocess.run(
    [sys.executable, '-c', 'import cudf, cuml; print(cudf.__version__)'],
    capture_output=True, text=True
)
if rapids_check.returncode == 0:
    print(f'RAPIDS already available (cudf {rapids_check.stdout.strip()}) — skipping pip install.')
    print('Proceed to the next cell without restarting the runtime.')
else:
    # Detect installed RAPIDS version to stay consistent; fall back to 26.2
    ver_check = subprocess.run(
        [sys.executable, '-c',
         'import importlib.metadata; print(importlib.metadata.version("libraft-cu12"))'],
        capture_output=True, text=True
    )
    rapids_ver = ver_check.stdout.strip().rsplit('.', 1)[0] if ver_check.returncode == 0 else '26.2'
    print(f'RAPIDS not found — installing cudf/cuml {rapids_ver}.*')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         '--extra-index-url', 'https://pypi.nvidia.com',
         f'cudf-cu12=={rapids_ver}.*', f'cuml-cu12=={rapids_ver}.*'],
        check=True
    )
    print('\nInstallation complete. RESTART THE RUNTIME now, then skip this cell.')

GPU OK: ['|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |', '| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |']
RAPIDS already available (cudf 26.02.01) — skipping pip install.
Proceed to the next cell without restarting the runtime.


### 1.2 Imports

A throwaway cuDF operation at the end forces CUDA context initialisation so the first real
cuDF call in section 3 is not penalised by driver startup latency.

In [ ]:
try:
    import cudf
    import cuml
except ImportError:
    raise RuntimeError('cuDF/cuML not found — run section 1.1 and restart the runtime.')

import numpy as np
import pandas as pd
import cudf
import cuml
from cuml.linear_model import LogisticRegression as cuLR

from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

import time
import json
import os
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print(f'cuDF  {cudf.__version__}  |  cuML  {cuml.__version__}')

# Force CUDA context initialisation — avoids attributing driver startup to section 3
_warmup = cudf.Series([1.0, 2.0, 3.0]).sum()
print('CUDA context initialised.')

cuDF  26.02.01  |  cuML  26.02.000
CUDA context initialised.


In [ ]:
# Persist outputs to Drive so they survive VM reclamation.
# To re-run from scratch: delete /content/drive/MyDrive/results/timings_gpu.csv first,
# otherwise Section 5 appends to the existing 90 rows and the sanity check in 5.4 fails.

from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.chdir('/content')   # keep working dir stable
if not os.path.exists('results'):
    os.symlink(RESULTS_DIR, 'results')
print(f'results/ now points to {RESULTS_DIR}')

Mounted at /content/drive
results/ now points to /content/drive/MyDrive/results


## 2. Data Loading & Train/Test Split

Identical `random_state=42` and `test_size=0.20` as `CPU_Baseline.ipynb`, guaranteeing
both notebooks operate on exactly the same rows.

In [ ]:
# REPRODUCIBILITY: this cell expects complaints_training.csv at
# /content/drive/MyDrive/complaints_training.csv (same path as CPU_Baseline.ipynb).
# best_params.json is loaded later in §4 from /content/drive/MyDrive/results/, so make
# sure CPU_Baseline.ipynb has been run first. All outputs (timings_gpu.csv and the
# updated best_params.json) will be written to /content/drive/MyDrive/results/ via
# the symlink set up in the cell above.

DATA_PATH = '/content/drive/MyDrive/complaints_training.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f'Loaded {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

Loaded 321,430 rows x 18 columns


In [ ]:
X = df_raw.drop(columns=['Complaint ID', 'Consumer disputed?'])
y = df_raw['Consumer disputed?'].map({'Yes': 1, 'No': 0})

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f'Train: {len(X_train_raw):,}  Test: {len(X_test_raw):,}')
print(f'Train positive rate: {y_train.mean():.3f}')

Train: 257,144  Test: 64,286
Train positive rate: 0.199


## 3. Feature Engineering

### 3.1 CPU Reference Function (not benchmarked)

Kept verbatim from `CPU_Baseline.ipynb`. Used only in section 3.3 to verify that the GPU
implementation produces numerically equivalent features.

In [ ]:
def feature_engineering(X, y=None, fit_encoders=None):
    encoders = {} if fit_encoders is None else fit_encoders
    training = fit_encoders is None
    raw = X.copy()
    for col in ['Complaint ID', 'Consumer disputed?']:
        if col in raw.columns:
            raw = raw.drop(columns=[col])
    out = pd.DataFrame(index=raw.index)

    date_rec  = pd.to_datetime(raw['Date received'],        errors='coerce')
    date_sent = pd.to_datetime(raw['Date sent to company'], errors='coerce')
    out['response_lag_days'] = (date_sent - date_rec).dt.days.clip(0, 120).fillna(0).astype(int)
    out['received_year']     = date_rec.dt.year.fillna(2015).astype(int)
    out['day_of_week']       = date_rec.dt.dayofweek.fillna(0).astype(int)

    resp = raw['Company response to consumer'].fillna('Missing')
    out['company_response_monetary']       = resp.str.contains('monetary',       case=False).astype(int)
    out['company_response_non_monetary']   = resp.str.contains('non-monetary',   case=False).astype(int)
    out['company_response_explanation']    = resp.str.contains('explanation',    case=False).astype(int)
    out['company_response_without_relief'] = resp.str.contains('without relief', case=False).astype(int)
    out['company_response_in_progress']    = resp.str.contains('in progress',    case=False).astype(int)
    out['company_response_is_relief']      = (
        (out['company_response_monetary'] == 1) |
        (out['company_response_non_monetary'] == 1)
    ).astype(int)
    out['timely_response']     = raw['Timely response?'].fillna('No').eq('Yes').astype(int)
    out['closed_x_timely']     = out['company_response_is_relief'] * out['timely_response']
    out['has_public_response'] = raw['Company public response'].notna().astype(int)

    narr = raw['Consumer complaint narrative'].fillna('')
    out['has_narrative']       = (narr != '').astype(int)
    out['word_count']          = narr.apply(lambda x: len(x.split()) if x else 0)
    out['char_count']          = narr.str.len().fillna(0).astype(int)
    out['avg_word_length']     = narr.apply(
        lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0
    )
    out['narrative_short']     = (out['word_count'].between(1, 19)).astype(int)
    out['narrative_long']      = (out['word_count'] > 200).astype(int)
    esc = 'attorney|lawyer|lawsuit|legal action|sue|court|cfpb|regulation|violation|fraud|illegal|discriminat'
    out['escalation_term_count'] = narr.str.lower().str.count(esc).fillna(0).astype(int)
    out['has_escalation_terms']  = (out['escalation_term_count'] > 0).astype(int)
    out['narrative_x_no_relief'] = out['has_narrative'] * (1 - out['company_response_is_relief'])

    consent = raw['Consumer consent provided?'].fillna('Missing')
    out['consent_provided']  = consent.eq('Consent provided').astype(int)
    out['consent_withdrawn'] = consent.eq('Consent withdrawn').astype(int)
    out['has_subproduct']    = raw['Sub-product'].notna().astype(int)
    out['has_subissue']      = raw['Sub-issue'].notna().astype(int)
    tags = raw['Tags'].fillna('')
    out['tags_any']          = (tags != '').astype(int)
    out['is_older_american'] = tags.str.contains('Older American', case=False).astype(int)
    out['is_servicemember']  = tags.str.contains('Servicemember',  case=False).astype(int)

    def target_encode(series, name, smoothing=20):
        if training:
            tmp = pd.DataFrame({'val': series.values, 'target': y.values}, index=series.index)
            agg = tmp.groupby('val')['target'].agg(['mean', 'count'])
            global_mean = y.mean()
            smooth = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
            encoders[name] = {'map': smooth.to_dict(), 'global_mean': global_mean}
        return series.map(encoders[name]['map']).fillna(encoders[name]['global_mean'])

    def freq_encode(series, name):
        if training:
            encoders[name] = series.value_counts().to_dict()
        return series.map(encoders[name]).fillna(1)

    out['Product_te']           = target_encode(raw['Product'].fillna('Missing'),                      'Product_te')
    out['Issue_te']             = target_encode(raw['Issue'].fillna('Missing'),                        'Issue_te')
    out['State_te']             = target_encode(raw['State'].fillna('Missing'),                        'State_te')
    out['Company_te']           = target_encode(raw['Company'].fillna('Missing'),                      'Company_te')
    out['Sub_product_te']       = target_encode(raw['Sub-product'].fillna('Missing'),                  'Sub_product_te')
    out['Sub_issue_te']         = target_encode(raw['Sub-issue'].fillna('Missing'),                    'Sub_issue_te')
    out['Submitted_via_te']     = target_encode(raw['Submitted via'].fillna('Missing'),                'Submitted_via_te')
    out['company_response_te']  = target_encode(raw['Company response to consumer'].fillna('Missing'), 'company_response_te')
    out['zip3_region_te']       = target_encode(
        raw['ZIP code'].fillna('000').astype(str).str[:3],                                             'zip3_region_te')
    prod_x_channel = raw['Product'].fillna('Missing') + '_' + raw['Submitted via'].fillna('Missing')
    out['product_x_channel_te'] = target_encode(prod_x_channel, 'product_x_channel_te')
    prod_x_issue = raw['Product'].fillna('Missing') + '_' + raw['Issue'].fillna('Missing')
    out['product_issue_te']     = target_encode(prod_x_issue, 'product_issue_te')
    out['company_volume']       = freq_encode(raw['Company'].fillna('Missing'), 'company_volume')
    out['response_te_x_relief'] = out['company_response_te'] * out['company_response_is_relief']
    return out, encoders

### 3.2 GPU Feature Engineering (`feature_engineering_gpu`)

Three systematic changes from the CPU version:

1. **`apply()` lambdas replaced with cuDF vectorised ops.** `narr.str.count(r'\S+')`
   replaces the Python-loop word count. cuDF does not support `apply()` with arbitrary
   Python callables.

2. **RE2-safe regex.** cuDF uses Google RE2, which rejects Perl-syntax lookaheads and some
   quantifiers. All `str.contains()` calls use `regex=False` (substring search) or simple
   alternation patterns RE2 accepts.

3. **Target encoding via cuDF groupby.** The smoothed mean is computed on-device; the
   resulting mapping dict is transferred back to host once and stored in `encoders` with
   the same schema as the CPU encoders, so `fit_encoders` is interchangeable.

The function returns a pandas DataFrame (`out.to_pandas()`). This boundary conversion is
intentional: downstream sklearn transformers (SimpleImputer, StandardScaler) and XGBoost
pipelines expect pandas or numpy. Keeping the FE output as pandas avoids silent type
errors and makes the cuDF/pandas boundary explicit.

In [ ]:
def feature_engineering_gpu(X, y=None, fit_encoders=None):
    encoders = {} if fit_encoders is None else fit_encoders
    training = fit_encoders is None

    X_copy = X.copy().reset_index(drop=True)

    # Compute all datetime-derived features on CPU before cuDF conversion.
    # pandas 2.x returns datetime64[us] (microsecond resolution) from to_datetime(),
    # and cuDF 26.x crashes when from_pandas() encounters that resolution —
    # it only handles datetime64[ns] reliably. Extracting the three numeric features
    # on CPU and dropping the raw datetime columns sidesteps this entirely.
    date_rec  = pd.to_datetime(X_copy['Date received'],        errors='coerce')
    date_sent = pd.to_datetime(X_copy['Date sent to company'], errors='coerce')
    _lag  = (date_sent - date_rec).dt.days.clip(0, 120).fillna(0).values.astype('int32')
    _year = date_rec.dt.year.fillna(2015).values.astype('int32')
    _dow  = date_rec.dt.dayofweek.fillna(0).values.astype('int32')
    X_copy = X_copy.drop(columns=['Date received', 'Date sent to company'], errors='ignore')

    # cudf.DataFrame.from_pandas() was removed in cuDF 26.x; use cudf.from_pandas()
    raw = cudf.from_pandas(X_copy)
    if training:
        y_gpu = cudf.Series(y.values)

    for col in ['Complaint ID', 'Consumer disputed?']:
        if col in raw.columns:
            raw = raw.drop(columns=[col])

    out = cudf.DataFrame()
    out['response_lag_days'] = cudf.Series(_lag)
    out['received_year']     = cudf.Series(_year)
    out['day_of_week']       = cudf.Series(_dow)

    resp = raw['Company response to consumer'].fillna('Missing')
    out['company_response_monetary']       = resp.str.contains('monetary',       regex=False).astype('int32')
    out['company_response_non_monetary']   = resp.str.contains('non-monetary',   regex=False).astype('int32')
    out['company_response_explanation']    = resp.str.contains('explanation',    regex=False).astype('int32')
    out['company_response_without_relief'] = resp.str.contains('without relief', regex=False).astype('int32')
    out['company_response_in_progress']    = resp.str.contains('in progress',    regex=False).astype('int32')
    out['company_response_is_relief']      = (
        (out['company_response_monetary'] == 1) |
        (out['company_response_non_monetary'] == 1)
    ).astype('int32')
    out['timely_response']     = (raw['Timely response?'].fillna('No') == 'Yes').astype('int32')
    out['closed_x_timely']     = out['company_response_is_relief'] * out['timely_response']
    out['has_public_response'] = (~raw['Company public response'].isna()).astype('int32')

    narr = raw['Consumer complaint narrative'].fillna('')
    out['has_narrative'] = (narr != '').astype('int32')
    # str.count(r'\S+') counts whitespace-delimited token runs — same as len(s.split())
    out['word_count']    = narr.str.count(r'\S+').fillna(0).astype('int32')
    out['char_count']    = narr.str.len().fillna(0).astype('int32')
    # avg_word_length: non-whitespace chars / word_count mirrors the CPU formula
    # np.mean([len(w) for w in x.split()]) — sum(len(w)) == count of \S chars,
    # divided by the number of tokens. Avoids the ±spaces approximation error.
    _non_ws = narr.str.count(r'\S').fillna(0)
    wc = out['word_count'].clip(lower=1)
    out['avg_word_length'] = (_non_ws / wc).fillna(0)
    out['narrative_short'] = (out['word_count'].between(1, 19)).astype('int32')
    out['narrative_long']  = (out['word_count'] > 200).astype('int32')
    esc = ('attorney|lawyer|lawsuit|legal action|sue|court|cfpb|'
           'regulation|violation|fraud|illegal|discriminat')
    narr_lower = narr.str.lower()
    out['escalation_term_count'] = narr_lower.str.count(esc).fillna(0).astype('int32')
    out['has_escalation_terms']  = (out['escalation_term_count'] > 0).astype('int32')
    out['narrative_x_no_relief'] = out['has_narrative'] * (1 - out['company_response_is_relief'])

    consent = raw['Consumer consent provided?'].fillna('Missing')
    out['consent_provided']  = (consent == 'Consent provided').astype('int32')
    out['consent_withdrawn'] = (consent == 'Consent withdrawn').astype('int32')
    out['has_subproduct']    = (~raw['Sub-product'].isna()).astype('int32')
    out['has_subissue']      = (~raw['Sub-issue'].isna()).astype('int32')
    tags = raw['Tags'].fillna('')
    out['tags_any']          = (tags != '').astype('int32')
    out['is_older_american'] = tags.str.contains('Older American', regex=False).astype('int32')
    out['is_servicemember']  = tags.str.contains('Servicemember',  regex=False).astype('int32')

    def target_encode_gpu(series, name, smoothing=20):
        series = series.reset_index(drop=True)
        if training:
            tmp = cudf.DataFrame({'val': series, 'target': y_gpu})
            agg = tmp.groupby('val')['target'].agg(['mean', 'count']).reset_index()
            agg.columns = ['val', 'mean', 'count']
            g = float(y_gpu.mean())
            agg['smooth'] = (agg['count'] * agg['mean'] + smoothing * g) / (agg['count'] + smoothing)
            encoders[name] = {
                'map': dict(zip(agg['val'].to_pandas(), agg['smooth'].to_pandas())),
                'global_mean': g
            }
        return series.map(encoders[name]['map']).fillna(encoders[name]['global_mean'])

    def freq_encode_gpu(series, name):
        series = series.reset_index(drop=True)
        if training:
            vc = series.value_counts()
            encoders[name] = dict(zip(vc.index.to_pandas(), vc.to_pandas()))
        return series.map(encoders[name]).fillna(1)

    out['Product_te']           = target_encode_gpu(raw['Product'].fillna('Missing'),                      'Product_te')
    out['Issue_te']             = target_encode_gpu(raw['Issue'].fillna('Missing'),                        'Issue_te')
    out['State_te']             = target_encode_gpu(raw['State'].fillna('Missing'),                        'State_te')
    out['Company_te']           = target_encode_gpu(raw['Company'].fillna('Missing'),                      'Company_te')
    out['Sub_product_te']       = target_encode_gpu(raw['Sub-product'].fillna('Missing'),                  'Sub_product_te')
    out['Sub_issue_te']         = target_encode_gpu(raw['Sub-issue'].fillna('Missing'),                    'Sub_issue_te')
    out['Submitted_via_te']     = target_encode_gpu(raw['Submitted via'].fillna('Missing'),                'Submitted_via_te')
    out['company_response_te']  = target_encode_gpu(raw['Company response to consumer'].fillna('Missing'), 'company_response_te')
    out['zip3_region_te']       = target_encode_gpu(
        raw['ZIP code'].fillna('000').astype(str).str.slice(0, 3),                                         'zip3_region_te')
    prod_x_channel = raw['Product'].fillna('Missing') + '_' + raw['Submitted via'].fillna('Missing')
    out['product_x_channel_te'] = target_encode_gpu(prod_x_channel, 'product_x_channel_te')
    prod_x_issue = raw['Product'].fillna('Missing') + '_' + raw['Issue'].fillna('Missing')
    out['product_issue_te']     = target_encode_gpu(prod_x_issue, 'product_issue_te')
    out['company_volume']       = freq_encode_gpu(raw['Company'].fillna('Missing'), 'company_volume')
    out['response_te_x_relief'] = out['company_response_te'] * out['company_response_is_relief']
    return out.to_pandas(), encoders

### 3.3 Correctness Verification: Feature Engineering

Run GPU FE on the full training and test sets, then compare every output column against
the CPU reference. We check that the maximum relative difference across all columns is
below `1e-4`. If any column exceeds this threshold the assert fails and the notebook stops.

The full-set GPU features (`X_train_enc_gpu`, `X_test_enc_gpu`) are retained in memory and
reused in sections 4.1 and 4.2.

In [ ]:
# GPU FE on full training and test sets — results kept for sections 4.1 and 4.2
X_train_enc_gpu, enc_gpu = feature_engineering_gpu(X_train_raw, y=y_train)
X_test_enc_gpu, _        = feature_engineering_gpu(X_test_raw, fit_encoders=enc_gpu)

# CPU FE for comparison only
X_train_enc_cpu, _ = feature_engineering(X_train_raw, y=y_train)

max_diff = 0.0
worst_col = ''

for col in X_train_enc_cpu.columns:
    cpu_v = X_train_enc_cpu[col].values.astype(float)
    gpu_v = X_train_enc_gpu[col].values.astype(float)
    rel = float(np.abs(cpu_v - gpu_v).max() / (np.abs(cpu_v).max() + 1e-9))
    if rel > max_diff:
        max_diff, worst_col = rel, col

print(f'Max rel diff across all columns: {max_diff:.2e}  (worst: {worst_col})  (threshold: 1e-4)')
assert max_diff < 1e-4, f'Column divergence too large — {worst_col}: {max_diff}'
print('FE correctness check PASSED.')
print(f'GPU FE shape: {X_train_enc_gpu.shape}')
del X_train_enc_cpu

Max rel diff across all columns: 0.00e+00  (worst: )  (threshold: 1e-4)
FE correctness check PASSED.
GPU FE shape: (257144, 41)


## 4. Hyperparameter Search (Context Timing)

Best hyperparameters are loaded from `results/best_params.json` (written by
`CPU_Baseline.ipynb`) so the GPU sweep uses the same model configuration as the CPU sweep.
The timing comparison measures the execution engine, not different models.

Two correctness checks run before the sweep:

- **4.1.** cuML LR AUC vs CPU LR AUC (threshold 0.002).
- **4.2.** GPU XGB AUC vs CPU XGB AUC (threshold 0.005), folded into the CV context timing.

Section 4.2 also runs the **H2D amortisation experiment**: the GPU XGB CV is executed
twice, once with a pandas input (host-to-device copy on every fit) and once with a
pre-loaded cuDF input (transfer paid once). Both timings are persisted in section 4.3.

In [ ]:
with open('results/best_params.json') as f:
    best_params = json.load(f)

print('Best LR params :', best_params['lr'])
print('Best XGB params:', best_params['xgb'])
print('CPU AUC  — LR :', best_params['auc']['lr'],
      '  XGB:', best_params['auc']['xgb'])
print('CPU CV context — LR :', round(best_params['cv_context_seconds']['lr_gridsearch'], 1), 's',
      '  XGB:', round(best_params['cv_context_seconds']['xgb_randomizedsearch'], 1), 's')

Best LR params : {'C': 0.1, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'lbfgs'}
Best XGB params: {'subsample': 0.65, 'scale_pos_weight': 4, 'reg_lambda': 5, 'reg_alpha': 0.1, 'n_estimators': 500, 'min_child_weight': 15, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.6}
CPU AUC  — LR : 0.6421907129973952   XGB: 0.6520096289775927
CPU CV context — LR : 138.1 s   XGB: 6079.9 s


### 4.1 Logistic Regression: AUC Equivalence Check

cuML LR has no sklearn-compatible cross-validation wrapper, so the context timing analogue
here is a single fit on the full training set rather than a full CV loop.

**Notes on the implementation:**

- `class_weight='balanced'` is not a cuML LR constructor argument. The equivalent is
  `sample_weight` passed to `.fit()`. `compute_sample_weight('balanced', ...)` from
  sklearn produces the same per-sample weights.
- sklearn's `SimpleImputer` and `StandardScaler` are used instead of cuML equivalents
  because they operate on numpy arrays (the output of `feature_engineering_gpu` is
  pandas), and their correctness relative to the CPU pipeline is already established.
  Adding cuML preprocessing would introduce another correctness surface without any
  benchmark benefit.
- AUC threshold is 0.002, tighter than XGB's 0.005, because L2 LR is a convex problem
  with a unique global minimum: GPU and CPU solvers should converge to effectively the
  same weights.

In [ ]:
best_lr_C       = best_params['lr']['C']
best_lr_penalty = best_params['lr'].get('penalty', 'l2')

imp_lr = SimpleImputer(strategy='median')
scl_lr = StandardScaler()
X_train_pp_full = scl_lr.fit_transform(imp_lr.fit_transform(X_train_enc_gpu)).astype(np.float32)
X_test_pp_full  = scl_lr.transform(imp_lr.transform(X_test_enc_gpu)).astype(np.float32)

sw_full = compute_sample_weight('balanced', y_train.values).astype(np.float32)
lr_full = cuLR(C=best_lr_C, penalty=best_lr_penalty, max_iter=1000)
lr_full.fit(X_train_pp_full, y_train.values.astype(np.float32), sample_weight=sw_full)

auc_lr_gpu  = roc_auc_score(y_test, lr_full.predict_proba(X_test_pp_full)[:, 1])
auc_lr_diff = abs(auc_lr_gpu - best_params['auc']['lr'])

print(f'GPU cuML LR  ROC-AUC : {auc_lr_gpu:.4f}')
print(f'CPU sklearn LR ROC-AUC: {best_params["auc"]["lr"]:.4f}')
print(f'AUC diff: {auc_lr_diff:.4f}  (threshold: 0.002)')
if auc_lr_diff >= 0.002:
    print(f'WARNING: LR AUC diff {auc_lr_diff:.4f} exceeds 0.002 threshold. '
          'Proceeding; document divergence in report.')
else:
    print('LR AUC correctness check PASSED.')

GPU cuML LR  ROC-AUC : 0.6422
CPU sklearn LR ROC-AUC: 0.6422
AUC diff: 0.0000  (threshold: 0.002)
LR AUC correctness check PASSED.


### 4.2 XGBoost: RandomizedSearchCV + H2D Amortisation

**Run 1, pandas input.** `X_train_enc_gpu` is a pandas DataFrame. XGBoost with
`device='cuda'` triggers a host-to-device (H2D) copy on every one of the 300 CV fits.
This is the default behaviour and the timing reported as the GPU CV context.

**Run 2, cuDF pre-loaded.** `X_train_enc_gpu` is converted to a cuDF DataFrame once before
the CV loop. XGBoost recognises a cuDF input as already device-resident and skips the H2D
copy on every fit, paying the transfer cost only once.

Both runs use identical search space, `n_iter=60`, and `random_state=42`, so the only
difference is the data transfer pattern. The speedup quantifies the cost of 299 redundant
H2D copies.

In [ ]:
param_grid_xgb = {
    'classifier__n_estimators':     [400, 500, 600, 800],
    'classifier__max_depth':        [4, 5, 6],
    'classifier__learning_rate':    [0.02, 0.03, 0.05],
    'classifier__scale_pos_weight': [3.5, 4, 4.5],
    'classifier__min_child_weight': [5, 10, 15],
    'classifier__subsample':        [0.55, 0.6, 0.65, 0.7],
    'classifier__colsample_bytree': [0.6, 0.65, 0.7],
    'classifier__reg_alpha':        [0, 0.1, 1],
    'classifier__reg_lambda':       [3, 5, 7]
}
cv_xgb = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Run 1: pandas input (H2D copy per fit) ────────────────────────────────────
xgb_pipe_pd = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('classifier', XGBClassifier(
        random_state=42, eval_metric='logloss',
        tree_method='hist', device='cuda'
    ))
])
search_pd = RandomizedSearchCV(
    xgb_pipe_pd, param_grid_xgb,
    n_iter=60, cv=cv_xgb, scoring='f1',
    random_state=42, n_jobs=1, verbose=1
)
t0 = time.perf_counter()
search_pd.fit(X_train_enc_gpu, y_train)
t_xgb_cv_pandas = time.perf_counter() - t0

y_prob_xgb_gpu = search_pd.best_estimator_.predict_proba(X_test_enc_gpu)[:, 1]
auc_xgb_gpu    = roc_auc_score(y_test, y_prob_xgb_gpu)
auc_xgb_diff   = abs(auc_xgb_gpu - best_params['auc']['xgb'])

print(f'[CONTEXT] GPU XGB CV (pandas, 300 fits): {t_xgb_cv_pandas:.1f}s  ({t_xgb_cv_pandas/60:.1f} min)')
print(f'Test ROC-AUC: {auc_xgb_gpu:.4f}  |  AUC diff vs CPU: {auc_xgb_diff:.4f}  (threshold: 0.005)')
if auc_xgb_diff >= 0.005:
    print(f'WARNING: XGB AUC diff {auc_xgb_diff:.4f} exceeds 0.005 threshold. '
          'Proceeding; document divergence in report.')
else:
    print('XGB AUC correctness check PASSED.')

# ── Run 2: cuDF pre-loaded (H2D once) ─────────────────────────────────────────
# Pre-impute on GPU with cuDF .fillna(median), then hand the cuDF frame directly
# to XGBClassifier. Putting sklearn's SimpleImputer in the pipeline would force a
# GPU to CPU conversion on every fit, defeating the purpose of staying on GPU.
# feature_engineering_gpu already produces NaN-free encoded columns; only the
# raw numeric columns need the median fill.
X_train_cudf = cudf.from_pandas(X_train_enc_gpu)

# Pre-impute on GPU. After this every column has dtype-appropriate fill.
medians = X_train_cudf.median()
X_train_cudf = X_train_cudf.fillna(medians)

# Single-step pipeline now that imputation is done; XGB takes the cuDF frame.
xgb_clf_cudf = XGBClassifier(
    random_state=42, eval_metric='logloss',
    tree_method='hist', device='cuda'
)
search_cudf = RandomizedSearchCV(
    xgb_clf_cudf, {k.replace('classifier__', ''): v for k, v in param_grid_xgb.items()},
    n_iter=60, cv=cv_xgb, scoring='f1',
    random_state=42, n_jobs=1, verbose=1
)
t0 = time.perf_counter()
search_cudf.fit(X_train_cudf, y_train)
t_xgb_cv_cudf = time.perf_counter() - t0

speedup_h2d = t_xgb_cv_pandas / t_xgb_cv_cudf
print(f'\n[H2D] GPU XGB CV (cuDF, 300 fits)       : {t_xgb_cv_cudf:.1f}s  ({t_xgb_cv_cudf/60:.1f} min)')
print(f'[H2D] Speedup from pre-loading to GPU   : {speedup_h2d:.2f}x')
print(f'[H2D] Time saved                        : {t_xgb_cv_pandas - t_xgb_cv_cudf:.1f}s')
del X_train_cudf

Fitting 5 folds for each of 60 candidates, totalling 300 fits
[CONTEXT] GPU XGB CV (pandas, 300 fits): 1047.5s  (17.5 min)
Test ROC-AUC: 0.6520  |  AUC diff vs CPU: 0.0000  (threshold: 0.005)
XGB AUC correctness check PASSED.
Fitting 5 folds for each of 60 candidates, totalling 300 fits

[H2D] GPU XGB CV (cuDF, 300 fits)       : 424.0s  (7.1 min)
[H2D] Speedup from pre-loading to GPU   : 2.47x
[H2D] Time saved                        : 623.5s


### 4.3 Persist GPU Context Timings

Update `best_params.json` in-place with two new keys read by `Analysis.ipynb`:

- `cv_context_seconds.xgb_randomizedsearch_gpu`: GPU CV time (pandas input, the standard).
- `h2d_amortisation`: both CV times for Figure 5 of the analysis notebook.

In [ ]:
best_params['cv_context_seconds']['xgb_randomizedsearch_gpu'] = t_xgb_cv_pandas
best_params['h2d_amortisation'] = {
    'xgb_cv_pandas_s': t_xgb_cv_pandas,
    'xgb_cv_cudf_s':   t_xgb_cv_cudf,
}

with open('results/best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)

print('Updated results/best_params.json')
print(f'  xgb_randomizedsearch_gpu : {t_xgb_cv_pandas:.1f}s')
print(f'  h2d_amortisation.pandas  : {t_xgb_cv_pandas:.1f}s')
print(f'  h2d_amortisation.cudf    : {t_xgb_cv_cudf:.1f}s')

Updated results/best_params.json
  xgb_randomizedsearch_gpu : 1047.5s
  h2d_amortisation.pandas  : 1047.5s
  h2d_amortisation.cudf    : 424.0s


## 5. Dataset-Size Sweep

A GPU warm-up pass runs before the first timed measurement. It exercises the same code
paths as the sweep (GPU FE, cuML LR fit/predict, GPU XGB fit/predict) on a small throwaway
subset, ensuring CUDA kernels are JIT-compiled and GPU memory pools are warm before any
timing begins. Without this, the first timing cell would absorb compilation latency that
does not recur on later iterations.

In [ ]:
print('Warm-up: running throwaway GPU ops to pre-compile CUDA kernels...')

_Xw = X_train_raw.sample(n=3000, random_state=0)
_yw = y_train.loc[_Xw.index]
_Xw_enc, _ = feature_engineering_gpu(_Xw, y=_yw)

# cuML LR warm-up
_imp = SimpleImputer(strategy='median')
_scl = StandardScaler()
_Xw_pp = _scl.fit_transform(_imp.fit_transform(_Xw_enc)).astype(np.float32)
_sw = compute_sample_weight('balanced', _yw.values).astype(np.float32)
_lr_w = cuLR(C=0.1, penalty='l2', max_iter=50)
_lr_w.fit(_Xw_pp, _yw.values.astype(np.float32), sample_weight=_sw)
_ = _lr_w.predict_proba(_Xw_pp)

# XGB GPU warm-up
_xgb_w = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('clf', XGBClassifier(n_estimators=10, tree_method='hist', device='cuda',
                          eval_metric='logloss', random_state=42))
])
_xgb_w.fit(_Xw_enc, _yw)
_ = _xgb_w.predict_proba(_Xw_enc)

print('Warm-up complete. Timing starts now.')

Warm-up: running throwaway GPU ops to pre-compile CUDA kernels...
Warm-up complete. Timing starts now.


### 5.1 Timing Utility

All timed calls go through `log_row` to guarantee schema consistency.

**Schema:** `device`, `fraction`, `n_rows`, `phase`, `run_idx`, `seconds`, `notes`.

In [ ]:
FRACS     = [0.10, 0.25, 0.50, 0.75, 1.00]
LR_HPO_FRACS  = [0.25, 0.50, 1.00]
XGB_HPO_FRACS = [0.50, 1.00]
RUNS      = 3
DEVICE    = 'gpu'

best_xgb_p = {k: v for k, v in best_params['xgb'].items()}

os.makedirs('results', exist_ok=True)
_rows = []

def log_row(device, fraction, n_rows, phase, run_idx, seconds, notes=''):
    _rows.append({
        'device': device, 'fraction': fraction, 'n_rows': n_rows,
        'phase': phase, 'run_idx': run_idx, 'seconds': seconds, 'notes': notes
    })

### 5.2 Fit/Predict Sweep

3 runs × 5 fractions × 6 phases = 90 rows, written to `results/timings_gpu.csv`. Mirrors
the CPU sweep exactly: same fractions, same `random_state=42` sampling, same phase names,
same best params. The CSV is saved after every outer `run_idx` iteration, so the run is
recoverable if Colab drops the session mid-sweep.

In [ ]:
# Reset state so re-running this cell produces a clean CSV.
_rows = []
_csv_path = 'results/timings_gpu.csv'
if os.path.exists(_csv_path):
    os.remove(_csv_path)
    print(f'Removed existing {_csv_path}; starting fresh sweep.')

for run_idx in range(RUNS):
    for frac in FRACS:
        X_sub = X_train_raw.sample(frac=frac, random_state=42)
        y_sub = y_train.loc[X_sub.index]
        n = len(X_sub)

        # fe_train
        t0 = time.perf_counter()
        X_sub_enc, enc_sub = feature_engineering_gpu(X_sub, y=y_sub)
        log_row(DEVICE, frac, n, 'fe_train', run_idx, time.perf_counter() - t0)

        # fe_test (fraction-specific encoders, measures real test-time FE overhead)
        t0 = time.perf_counter()
        X_test_sub, _ = feature_engineering_gpu(X_test_raw, fit_encoders=enc_sub)
        log_row(DEVICE, frac, n, 'fe_test', run_idx, time.perf_counter() - t0)

        # lr_fit. cuML LR; sklearn imputer+scaler match CPU pipeline structure
        t0 = time.perf_counter()
        imp = SimpleImputer(strategy='median')
        scl = StandardScaler()
        X_sub_pp = scl.fit_transform(imp.fit_transform(X_sub_enc)).astype(np.float32)
        sw = compute_sample_weight('balanced', y_sub.values).astype(np.float32)
        lr_gpu = cuLR(C=best_lr_C, penalty=best_lr_penalty, max_iter=1000)
        lr_gpu.fit(X_sub_pp, y_sub.values.astype(np.float32), sample_weight=sw)
        log_row(DEVICE, frac, n, 'lr_fit', run_idx, time.perf_counter() - t0)

        # lr_predict (test preprocessing included to match CPU timing scope)
        t0 = time.perf_counter()
        X_test_pp = scl.transform(imp.transform(X_test_sub)).astype(np.float32)
        lr_gpu.predict_proba(X_test_pp)
        log_row(DEVICE, frac, n, 'lr_predict', run_idx, time.perf_counter() - t0)

        # xgb_fit
        xgb_gpu = Pipeline([
            ('imputer',    SimpleImputer(strategy='median')),
            ('classifier', XGBClassifier(
                **best_xgb_p,
                tree_method='hist', device='cuda',
                random_state=42, eval_metric='logloss'
            ))
        ])
        t0 = time.perf_counter()
        xgb_gpu.fit(X_sub_enc, y_sub)
        log_row(DEVICE, frac, n, 'xgb_fit', run_idx, time.perf_counter() - t0)

        # xgb_predict
        t0 = time.perf_counter()
        xgb_gpu.predict_proba(X_test_sub)
        log_row(DEVICE, frac, n, 'xgb_predict', run_idx, time.perf_counter() - t0)

        print(f'run={run_idx} frac={frac:.0%} n={n:,} done')

    pd.DataFrame(_rows).to_csv('results/timings_gpu.csv', index=False)
    print(f'[run {run_idx}] Saved results/timings_gpu.csv  ({len(_rows)} rows so far)')

run=0 frac=10% n=25,714 done
run=0 frac=25% n=64,286 done
run=0 frac=50% n=128,572 done
run=0 frac=75% n=192,858 done
run=0 frac=100% n=257,144 done
[run 0] Saved results/timings_gpu.csv  (30 rows so far)
run=1 frac=10% n=25,714 done
run=1 frac=25% n=64,286 done
run=1 frac=50% n=128,572 done
run=1 frac=75% n=192,858 done
run=1 frac=100% n=257,144 done
[run 1] Saved results/timings_gpu.csv  (60 rows so far)
run=2 frac=10% n=25,714 done
run=2 frac=25% n=64,286 done
[2026-04-25 14:02:12.256] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
run=2 frac=50% n=128,572 done
run=2 frac=75% n=192,858 done
run=2 frac=100% n=257,144 done
[run 2] Saved results/timings_gpu.csv  (90 rows so far)


### 5.3 HPO Sweep

3 fractions for `lr_cv` (25%, 50%, 100%) + 2 fractions for `xgb_cv` (50%, 100%) = **5 HPO
rows** appended to the same `_rows` list and CSV as the fit/predict sweep above.

**`lr_cv`.** cuML LR has no sklearn-compatible CV wrapper, so the CV is hand-rolled around
cuML LR (`cuml_lr_grid_cv` below): it walks the `C` grid over 5 stratified folds and times
the whole loop. This keeps every fit on GPU, which is what we want to measure.

**`xgb_cv`.** `RandomizedSearchCV` with `XGBClassifier(device='cuda')`. GPU acceleration
applies to every tree-fit inside the CV loop. The 100% fraction reuses the timing already
collected in §4.2 (same search, same data) to avoid ~25 minutes of redundant GPU work.

The reload guard at the top protects against kernel restarts: if `_rows` was cleared, it
is repopulated from the CSV written by section 5.2 before new rows are appended.

In [ ]:
# Reload from CSV if _rows was lost in a kernel restart
if len(_rows) < RUNS * len(FRACS) * 6:
    _rows = pd.read_csv('results/timings_gpu.csv').to_dict('records')
    print(f'Reloaded {len(_rows)} rows from timings_gpu.csv')

from sklearn.metrics import f1_score

# Hand-rolled 5-fold CV around cuML LR, mirrors sklearn GridSearchCV(scoring='f1').
# cuML has no native GridSearchCV; wrapping sklearn's would force CPU fits, which is
# exactly what we are trying NOT to measure in the GPU notebook.
def cuml_lr_grid_cv(X_df, y_arr, Cs, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    X_vals = X_df.values if hasattr(X_df, 'values') else X_df
    best_score, best_C = -np.inf, None
    for C in Cs:
        fold_scores = []
        for tr_idx, va_idx in skf.split(X_vals, y_arr):
            X_tr, X_va = X_vals[tr_idx], X_vals[va_idx]
            y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]
            imp = SimpleImputer(strategy='median')
            scl = StandardScaler()
            X_tr_pp = scl.fit_transform(imp.fit_transform(X_tr)).astype(np.float32)
            X_va_pp = scl.transform(imp.transform(X_va)).astype(np.float32)
            sw = compute_sample_weight('balanced', y_tr).astype(np.float32)
            m = cuLR(C=C, penalty='l2', max_iter=1000)
            m.fit(X_tr_pp, y_tr.astype(np.float32), sample_weight=sw)
            preds = (m.predict_proba(X_va_pp)[:, 1] > 0.5).astype(int)
            fold_scores.append(f1_score(y_va, preds))
        mean = float(np.mean(fold_scores))
        if mean > best_score:
            best_score, best_C = mean, C
    return best_score, best_C

LR_Cs = [0.001, 0.01, 0.1, 1, 10]  # same grid as CPU 4.1

# ── lr_cv: cuML LR at each LR_HPO_FRACS ────────────────────────────────
for frac in LR_HPO_FRACS:
    X_sub = X_train_raw.sample(frac=frac, random_state=42)
    y_sub = y_train.loc[X_sub.index]
    n     = len(X_sub)
    X_sub_enc, _ = feature_engineering_gpu(X_sub, y=y_sub)

    t0 = time.perf_counter()
    _ = cuml_lr_grid_cv(X_sub_enc, y_sub.values, LR_Cs, n_splits=5)
    t_this = time.perf_counter() - t0
    log_row(DEVICE, frac, n, 'lr_cv', 0, t_this,
            notes=f'{len(LR_Cs) * 5} fits; hand-rolled cuML LR CV')
    print(f'lr_cv  frac={frac:.0%}  n={n:,}  elapsed={t_this/60:.1f} min')

    pd.DataFrame(_rows).to_csv('results/timings_gpu.csv', index=False)

# ── xgb_cv: RandomizedSearchCV with device='cuda' ──────────────────────
param_grid_xgb_hpo = {
    'classifier__n_estimators':     [400, 500, 600, 800],
    'classifier__max_depth':        [4, 5, 6],
    'classifier__learning_rate':    [0.02, 0.03, 0.05],
    'classifier__scale_pos_weight': [3.5, 4, 4.5],
    'classifier__min_child_weight': [5, 10, 15],
    'classifier__subsample':        [0.55, 0.6, 0.65, 0.7],
    'classifier__colsample_bytree': [0.6, 0.65, 0.7],
    'classifier__reg_alpha':        [0, 0.1, 1],
    'classifier__reg_lambda':       [3, 5, 7]
}

for frac in XGB_HPO_FRACS:
    X_sub = X_train_raw.sample(frac=frac, random_state=42)
    y_sub = y_train.loc[X_sub.index]
    n     = len(X_sub)

    # Reuse 4.2's pandas-input timing for frac=1.00: same search, same data, already ran.
    # Avoids ~25 min of redundant GPU work.
    if frac == 1.00:
        log_row(DEVICE, 1.00, n, 'xgb_cv', 0, t_xgb_cv_pandas,
                notes='300 fits; reused from 4.2 pandas-input run')
        print(f'xgb_cv frac={frac:.0%}  n={n:,}  reused 4.2  '
              f'elapsed={t_xgb_cv_pandas/60:.1f} min')
        pd.DataFrame(_rows).to_csv('results/timings_gpu.csv', index=False)
        continue

    X_sub_enc, _ = feature_engineering_gpu(X_sub, y=y_sub)

    xgb_hpo = Pipeline([
        ('imputer',    SimpleImputer(strategy='median')),
        ('classifier', XGBClassifier(
            random_state=42, eval_metric='logloss',
            tree_method='hist', device='cuda'
        )),
    ])
    search_xgb_hpo = RandomizedSearchCV(
        xgb_hpo, param_grid_xgb_hpo,
        n_iter=60, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='f1', random_state=42, n_jobs=1, verbose=0
    )
    t0 = time.perf_counter()
    search_xgb_hpo.fit(X_sub_enc, y_sub)
    t_this = time.perf_counter() - t0
    log_row(DEVICE, frac, n, 'xgb_cv', 0, t_this,
            notes=f'{search_xgb_hpo.n_iter * 5} fits')
    print(f'xgb_cv frac={frac:.0%}  n={n:,}  elapsed={t_this/60:.1f} min')

    pd.DataFrame(_rows).to_csv('results/timings_gpu.csv', index=False)

print('HPO sweep complete.')

[2026-04-25 14:02:38.655] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
lr_cv  frac=25%  n=64,286  elapsed=0.1 min
[2026-04-25 14:02:55.452] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-04-25 14:02:59.426] [CUML] [warning] L-BFGS stopped, because the line search failed to advance (step delta = 0.000000)
lr_cv  frac=50%  n=128,572  elapsed=0.3 min
[2026-04-25 14:03:20.364] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-04-25 14:03:23.166] [CUML] [warning] L-BFGS stopped, because the line search failed to advance (step delta = 0.000000)
[2026-04-25 14:03:28.795] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-04-25 14:03:30.679] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-04-25 14:03:35.424] [CUML] [warning] L-BFGS stopped, because the line search failed to advance (step d

### 5.4 Sanity Check

Verify row counts before handing the CSV to `Analysis.ipynb`:
- Main sweep: 3 runs × 5 fractions × 6 phases = **90 rows**
- HPO sweep: 3 `lr_cv` rows (25%, 50%, 100%) + 2 `xgb_cv` rows (50%, 100%) = **5 rows**

In [ ]:
df_gpu = pd.read_csv('results/timings_gpu.csv')

MAIN_PHASES = ['fe_train', 'fe_test', 'lr_fit', 'lr_predict', 'xgb_fit', 'xgb_predict']
HPO_PHASES  = ['lr_cv', 'xgb_cv']

main_rows = df_gpu[df_gpu['phase'].isin(MAIN_PHASES)]
hpo_rows  = df_gpu[df_gpu['phase'].isin(HPO_PHASES)]

expected_main = RUNS * len(FRACS) * 6
expected_hpo  = len(LR_HPO_FRACS) + len(XGB_HPO_FRACS)   # 3 + 2 = 5

print(f'Main sweep rows : {len(main_rows):>4}  (expected {expected_main})')
print(f'HPO sweep rows  : {len(hpo_rows):>4}  (expected {expected_hpo})')

assert len(main_rows) == expected_main, f'Main sweep: expected {expected_main}, got {len(main_rows)}'
assert len(hpo_rows)  == expected_hpo,  f'HPO sweep: expected {expected_hpo}, got {len(hpo_rows)}'
print('Row counts OK.')
print()
print(main_rows.groupby(['phase', 'fraction'])['seconds'].median().unstack().to_string())

Main sweep rows :   90  (expected 90)
HPO sweep rows  :    5  (expected 5)
Row counts OK.

fraction         0.10      0.25      0.50      0.75      1.00
phase                                                        
fe_test      1.141435  1.262387  1.154946  1.248101  1.149812
fe_train     0.593297  1.288061  2.192109  3.278176  4.179593
lr_fit       0.122203  0.396795  0.716817  1.472909  1.480668
lr_predict   0.038121  0.043148  0.039929  0.044574  0.038782
xgb_fit      0.951181  1.762468  2.217540  3.369981  4.615308
xgb_predict  0.101411  0.107857  0.106666  0.102390  0.151006


In [ ]:
import pandas as pd
df = pd.read_csv('results/timings_gpu.csv')

# Show only the HPO rows
hpo = df[df['phase'].isin(['lr_cv', 'xgb_cv'])]
print(hpo[['phase', 'fraction', 'n_rows', 'seconds', 'notes']].to_string(index=False))

 phase  fraction  n_rows     seconds                                       notes
 lr_cv      0.25   64286    7.513595             25 fits; hand-rolled cuML LR CV
 lr_cv      0.50  128572   18.043114             25 fits; hand-rolled cuML LR CV
 lr_cv      1.00  257144   37.955590             25 fits; hand-rolled cuML LR CV
xgb_cv      0.50  128572  579.072195                                    300 fits
xgb_cv      1.00  257144 1047.520310 300 fits; reused from §4.2 pandas-input run
